In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 112 not upgraded.


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!ollama --version

In [5]:
import os
os.environ["OLLAMA_MODELS"]="/content/drive/MyDrive/LLM_Server/models"

In [6]:
!nohup ollama serve > /content/ollama.log 2>&1 &

In [7]:
!ollama pull qwen2.5:7b

In [8]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


In [9]:
!du -sh ~/.ollama

32K	/root/.ollama


In [10]:
!chmod +x /content/drive/MyDrive/LLM_Server/scripts/start_ollama.sh

In [11]:
!/content/drive/MyDrive/LLM_Server/scripts/start_ollama.sh

Ollama Started
NAME          ID              SIZE      MODIFIED      
qwen2.5:7b    845dbda0ea48    4.7 GB    6 seconds ago    


In [12]:
!pip install fastapi uvicorn requests

In [13]:
%%writefile /content/drive/MyDrive/LLM_Server/server.py

from fastapi import FastAPI
import requests

app = FastAPI()


OLLAMA_URL = "http://127.0.0.1:11434"


@app.get("/")
def home():
    return {
        "status": "online",
        "service": "Colab LLM API"
    }


@app.get("/health")
def health():
    return {
        "status": "running"
    }


@app.get("/models")
def models():

    try:
        r = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=10
        )

        return r.json()

    except Exception as e:
        return {
            "error": str(e)
        }


@app.post("/chat")
def chat(data: dict):

    prompt = data.get("prompt")

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()

Overwriting /content/drive/MyDrive/LLM_Server/server.py


In [14]:
!pkill uvicorn

%cd /content/drive/MyDrive/LLM_Server

!nohup uvicorn server:app \
--host 0.0.0.0 \
--port 8000 \
> /content/api.log 2>&1 &

/content/drive/MyDrive/LLM_Server


In [15]:
!cat /content/api.log

In [16]:
!curl http://127.0.0.1:8000/health

curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Connection refused


In [17]:
!pip install websockets aiohttp

In [18]:
%%writefile /content/drive/MyDrive/LLM_Server/relay_server.py

import asyncio
import websockets
import json
import aiohttp


CLIENTS = set()


async def handler(websocket):

    print("Client connected")

    CLIENTS.add(websocket)

    try:
        async for message in websocket:
            print("Received:", message)

    except Exception as e:
        print(e)

    finally:
        CLIENTS.remove(websocket)


async def main():

    server = await websockets.serve(
        handler,
        "0.0.0.0",
        8765
    )

    print("Relay running on 8765")

    await server.wait_closed()


asyncio.run(main())

Overwriting /content/drive/MyDrive/LLM_Server/relay_server.py


In [19]:
!nohup python /content/drive/MyDrive/LLM_Server/relay_server.py > /content/relay.log 2>&1 &

In [29]:
!nohup ssh -o StrictHostKeyChecking=no \
-o ServerAliveInterval=60 \
-R 80:127.0.0.1:8000 \
nokey@localhost.run > /content/tunnel.log 2>&1 &

In [30]:
!cat /content/tunnel.log

Pseudo-terminal will not be allocated because stdin is not a terminal.

Welcome to localhost.run!

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/


** your connection id is 34.126.135.195:43398, please mention it if you send me a message about an issue. **


2f6711387be324.lhr.life tunneled with tls termination, https://2f6711387be324.lhr.life
create an account and add your key for a longer lasting domain name. see https://localhost.run/docs/forever-free/ for more information.
Open your tunnel address on your mobile with this QR:

                                        

In [31]:
!curl http://127.0.0.1:8000/health

{"status":"running"}

In [32]:
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"qwen2.5:7b","model":"qwen2.5:7b","modified_at":"2026-09-03T11:45:14Z","size":4683087332,"digest":"845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":3584},"capabilities":["completion","tools"]}]}

In [33]:
!cat /content/api.log

INFO:     Started server process [32009]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:47762 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:44470 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:60626 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:60642 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:60762 - "POST //chat HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:58494 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:59444 - "GET /health HTTP/1.1" 200 OK
